# TCT nickel wastewater

See the repository README and reproduction guide for data requirements and experiment settings. Generated models and histories are written to `outputs/tct_nickel_wastewater/`.


In [ ]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "research_paths.py").is_file()
)
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))
from research_paths import data_file, external_file, checkpoint_file, history_file, output_file


In [ ]:
seed_value = 777

import numpy as np
import random
import tensorflow as tf

# Set random seeds for reproducibility
np.random.seed(seed_value)
random.seed(seed_value)
tf.random.set_seed(seed_value)

# For PyTorch, if used (commented if not applicable)
try:
    import torch

    torch.manual_seed(seed_value)
    torch.cuda.manual_seed(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
except ImportError:
    pass

# Fixing thread settings to enforce determinism (optional)
import os

os.environ["TF_DETERMINISTIC_OPS"] = "1"

# Explicitly set initializers for TensorFlow/Keras models
from tensorflow.keras import initializers

initializer = initializers.GlorotUniform(seed=seed_value)


import numpy as np
import random
import tensorflow as tf

# Set random seeds for reproducibility
np.random.seed(seed_value)
random.seed(seed_value)
tf.random.set_seed(seed_value)

# For PyTorch, if used (commented if not applicable)
try:
    import torch

    torch.manual_seed(seed_value)
    torch.cuda.manual_seed(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
except ImportError:
    pass

# For scikit-learn, ensure random_state is set where applicable


import numpy as np
import random
import tensorflow as tf

# Set random seeds for reproducibility
np.random.seed(seed_value)
random.seed(seed_value)
tf.random.set_seed(seed_value)


import numpy as np
import random

# Set random seeds for reproducibility
np.random.seed(seed_value)
random.seed(seed_value)

import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import pandas as pd
import os
import csv


In [ ]:
"""
LOAD TRAINING DATA
"""

# Load training data
path = data_file("spectra/recollected_training.csv")

# Read the data from CSV
data = pd.read_csv(path, dtype=float)
col = list(data.columns)

# Extract wavelengths
wavelength = np.array([float(j) for j in col[18:1880]])

# Convert data to numpy array
data = np.array(data)

# Extract specific columns for labels
ontime = data[:, col.index("Ontime")]
conductivity = data[:, col.index("Conductivity")]
concentration = data[:, col.index("Ni")]

# Extract spectrum containing only spectrum information
spectrum = data[:, 18:1880]
spectrum = np.clip(spectrum, None, 60000)

# Indices to be removed
Pb0_index = np.array([col.index("278.051"), col.index("286.721")]) - 18
Pb1_index = np.array([col.index("360.189"), col.index("375.312")]) - 18
Pb2_index = np.array([col.index("400.718"), col.index("410.412")]) - 18
Zn1_index = np.array([col.index("210.081"), col.index("220.118")]) - 18
Zn2_index = np.array([col.index("468.665"), col.index("485.098")]) - 18
Cu_index = np.array([col.index("323.137"), col.index("340.195")]) - 18

# Collect indices to be removed
zero_index = np.concatenate(
    [
        np.arange(Pb0_index[0], Pb0_index[-1] + 1),
        np.arange(Pb1_index[0], Pb1_index[-1] + 1),
        np.arange(Pb2_index[0], Pb2_index[-1] + 1),
        np.arange(Cu_index[0], Cu_index[-1] + 1),
        np.arange(Zn1_index[0], Zn1_index[-1] + 1),
        np.arange(Zn2_index[0], Zn2_index[-1] + 1),
    ]
)

# Set specific indices to zero
spectrum[:, zero_index] = 0
spectrum = spectrum / 60000

# Dimensions and number of spectra
dimension = spectrum.shape[1]
training_number = spectrum.shape[0]

print("\n" + "=" * 40 + "\n")
print("Spectra Dimension - final", dimension)
print("Training Spectra number - final", training_number)
print("Training Data shape", spectrum.shape)
print("Training Label shape", concentration.shape)
print("\n" + "=" * 40 + "\n")


In [ ]:
"""
LOAD TESTING DATA
"""

path = data_file("spectra/recollected_testing.csv")
# data not in np.array() form (no wavelength)
data_testing = pd.read_csv(path, dtype=float)
# data in np.array() form (no wavelength)
data_testing = np.array(data_testing)
# Ontime
ontime_test = data_testing[:, col.index("Ontime")]
# Conductivity
conductivity_test = data_testing[:, col.index("Conductivity")]
# Concentration labels in ppm (target element selected below).
concentration_test = data_testing[:, col.index("Ni")]
# spectrum_test containing only spectrum information
spectrum_test = data_testing[:, 18:1880]
spectrum = np.clip(spectrum, None, 60000)
spectrum_test[:, zero_index] = 0
spectrum_test = spectrum_test / 60000
dimension = len(spectrum_test[0])
testing_number = len(spectrum_test)
print("\n" + "=" * 40 + "\n")
print("Sprctra Dimension - final", dimension)
print("Testing Spectra number - final", testing_number)
print("Testing Data shape", spectrum_test.shape)
print("Testing Label shape", concentration_test.shape)
print("\n" + "=" * 40 + "\n")


In [ ]:
plt.plot(wavelength, spectrum[0])
nonzero_count = sum(1 for w in spectrum[0] if w != 0)
print(nonzero_count)


In [ ]:
ontime = ontime.reshape(len(ontime), 1)
conductivity = conductivity.reshape(len(conductivity), 1)
concentration = concentration.reshape(len(concentration), 1)
ontime_test = ontime_test.reshape(len(ontime_test), 1)
conductivity_test = conductivity_test.reshape(len(conductivity_test), 1)
concentration_test = concentration_test.reshape(len(concentration_test), 1)
# Stack the arrays horizontally
training = spectrum
testing = spectrum_test


In [ ]:
import tensorflow.keras as tfkeras
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow.keras.backend as K
from sklearn.model_selection import train_test_split
from matplotlib.pyplot import colorbar
from sklearn.utils import shuffle
from tensorflow.keras import regularizers
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix
import numpy as np

# Training parameters
learning_rate = 0.00004
opt = tfkeras.optimizers.Adam(learning_rate=learning_rate)
# early_stop = tfkeras.callbacks.EarlyStopping(monitor='val_mae', patience=3000,verbose=0, mode='min',restore_best_weights='True')


In [ ]:
## from matplotlib.pyplot import colorbar1
from sklearn.utils import shuffle
from tensorflow.keras import regularizers
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from keras.layers import LSTM, Dense

dim = int(np.max(ontime) + 1)
input_spectrum = tf.keras.Input(shape=(training.shape[1],))
input_conductivity = tf.keras.Input(shape=(1,))
input_ontime = tf.keras.Input(shape=(1,))

# Reshape input_spectrum
spectrum_shape = layers.Reshape((training.shape[1], 1))(input_spectrum)
conv1 = layers.Conv1D(32, 5, kernel_regularizer=regularizers.l2(0.001))(spectrum_shape)
conv1 = layers.MaxPooling1D(3)(conv1)
conv2 = layers.Conv1D(32, 5, kernel_regularizer=regularizers.l2(0.001))(conv1)
conv2 = layers.MaxPooling1D(3)(conv2)

# Transformer Encoder Block
# Multi-head Attention
attention_output = layers.MultiHeadAttention(num_heads=4, key_dim=64)(conv2, conv2)
attention_output = layers.Dropout(0.2)(attention_output)
attention_output = layers.LayerNormalization(epsilon=1e-6)(attention_output + conv2)  # Add & Norm

# Feed-Forward Network (FFN)
ffn = layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(0.001))(
    attention_output
)
ffn = layers.Dropout(0.1)(ffn)
ffn_output = layers.LayerNormalization(epsilon=1e-6)(ffn + attention_output)  # Add & Norm

# Flatten and Dense Layers for Final Prediction
flatten = layers.Flatten()(ffn_output)
dense = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(0.001))(flatten)
dense = layers.Dropout(0.2)(dense)
dense = layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(0.001))(dense)
dense = layers.Dropout(0.2)(dense)
output = layers.Dense(1)(dense)

# Build Model
TCT_Model = Model(inputs=input_spectrum, outputs=output, name="Temporal_Convolutional_Transformer")
TCT_Model.summary()


def smape(y_true, y_pred):
    epsilon = tf.keras.backend.epsilon()  # Small constant to avoid division by zero
    numerator = tf.abs(y_pred - y_true)
    denominator = tf.abs(y_true) + tf.abs(y_pred)
    smape_loss = tf.where(
        tf.equal(y_true, 0),
        numerator,  # Use absolute error when y_true is zero
        numerator / (denominator + epsilon) * 2,  # Use SMAPE otherwise
    )
    return tf.reduce_mean(smape_loss) * 100


# Compile and train the model
TCT_Model.compile(loss=smape, optimizer=opt, metrics=["mae"])


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import Callback, EarlyStopping


# Custom validation callback
class MultiValidationCallback(Callback):
    def __init__(self, model, validation_data_2):
        self.model = model
        self.validation_data_2 = validation_data_2

    def on_epoch_end(self, epoch, logs=None):
        X_val2, Y_val2 = self.validation_data_2
        val_preds_2 = self.model.predict(X_val2, verbose=0)

        # Calculate SMAPE and MAPE
        val_loss_2, val_mae_2 = self.model.evaluate(X_val2, Y_val2, verbose=0)
        mape_2 = np.mean(np.abs((Y_val2 - val_preds_2) / (Y_val2 + 1e-8))) * 100
        print(
            f"Epoch {epoch+1}: Validation 2 Loss = {val_loss_2:.4f}, Validation 2 MAE = {val_mae_2:.4f}, Validation 2 MAPE = {mape_2:.4f}%"
        )


# Split training and validation data
X_train, X_val, Y_train, Y_val = train_test_split(
    training, concentration, test_size=0.1, random_state=seed_value
)

# Configure the two validation datasets
validation_data_1 = (X_val, Y_val)
validation_data_2 = (testing, concentration_test)

# Early stopping based on validation_data_1
early_stop = EarlyStopping(
    monitor="val_loss",  # Monitor loss on the first validation dataset
    patience=500,  # Stop after the configured patience without improvement
    restore_best_weights=True,  # Restore the best model weights
)

# Initialize the callback
multi_val_callback = MultiValidationCallback(model=TCT_Model, validation_data_2=validation_data_2)

# Train the model
train_history = TCT_Model.fit(
    X_train,
    Y_train,
    validation_data=validation_data_1,  # Use the first dataset for internal validation
    batch_size=32,
    epochs=20000,
    verbose=2,
    callbacks=[early_stop, multi_val_callback],  # Include early stopping and custom validation
)

# Record training and validation history
Epoch = len(train_history.history["mae"])
Training_History = np.zeros((4, Epoch))
Training_History[0, :] = np.array(train_history.history["loss"])
Training_History[1, :] = np.array(train_history.history["mae"])
Training_History[2, :] = np.array(train_history.history["val_loss"])
Training_History[3, :] = np.array(train_history.history["val_mae"])

# Save training history
Array_name = str(output_file("Cu_Transfer.npy", "tct_nickel_wastewater"))
np.save(Array_name, Training_History)

# Save the model
Model_name = str(output_file("Cu_Transfer.h5", "tct_nickel_wastewater"))
TCT_Model.save(Model_name)


In [ ]:
# Training
Y_pred = TCT_Model.predict([training])
plt.plot(Y_pred, ".")
plt.plot(concentration)
plt.xlabel("number of training data")
plt.ylabel("ppm")
plt.show()
print(tf.keras.losses.MeanSquaredError()(concentration, Y_pred))


In [ ]:
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

plt.plot(Training_History[1, :], color="blue", label="Training mae")
plt.plot(Training_History[3, :], color="red", label="Val mae")
plt.legend()
plt.xlabel("epochs")
plt.ylabel("mae")
plt.show()
Y_pred = TCT_Model.predict([testing])
print(tf.keras.losses.MeanSquaredError()(concentration_test, Y_pred))

# Testing
plt.plot(range(0, 30), Y_pred[:30], ".", color="orange", alpha=0.7)
plt.plot(range(30, 60), Y_pred[30:60], ".", color="m", alpha=0.7)
plt.plot(range(60, 90), Y_pred[60:90], ".", color="orange", alpha=0.7)
plt.plot(range(90, 120), Y_pred[90:120], ".", color="m", alpha=0.7)
plt.plot(range(120, 150), Y_pred[120:150], ".", color="orange", alpha=0.7)
plt.plot(range(150, 180), Y_pred[150:180], ".", color="m", alpha=0.7)
plt.plot(range(180, 210), Y_pred[180:210], ".", color="orange", alpha=0.7)
plt.ylim(0, 22)
plt.plot(concentration_test, color="black", linewidth=3.0)
plt.xlabel("number of testing data", fontsize=12)
plt.ylabel("Concentration (ppm)", fontsize=12)
plt.title("Predictions vs Actual Concentrations", fontsize=14)

# Add legend
orange_patch = mpatches.Patch(color="orange", label="7.5")
magenta_patch = mpatches.Patch(color="m", label="15")
plt.legend(handles=[orange_patch, magenta_patch], fontsize=10, loc="upper right")
# Add arrow pointing to the right
plt.annotate(
    "", xy=(300, 1), xytext=(0, 1), arrowprops=dict(facecolor="black", arrowstyle="->", linewidth=3)
)
plt.text(150, 2, "Conductivity Increases", fontsize=16, ha="center")
plt.show()
Y_pred = TCT_Model.predict([testing])
print(tf.keras.losses.MeanSquaredError()(concentration_test, Y_pred))


In [ ]:
# Calculate MAE and display three decimal places
mae = tf.keras.losses.MeanAbsoluteError()
mae_value = mae(concentration_test, Y_pred).numpy()
print(f"{mae_value:.3f}")

# Calculate MAPE and display three decimal places with a percent sign
mape = tf.keras.losses.MeanAbsolutePercentageError()
mape_value = mape(concentration_test, Y_pred).numpy()
print(f"{mape_value:.3f}%")


In [ ]:
"""
LOAD WASTEWATER DATA
"""

path = data_file("spectra/wastewater_combined.csv")
# data not in np.array() form (no wavelength)
data_wastewater = pd.read_csv(path, dtype=float)
# data in np.array() form (no wavelength)
data_wastewater = np.array(data_wastewater)

# Cu concentration labels in ppm.
concentration_wastewater = data_wastewater[:, col.index("Cu")]
# spectrum_test containing only spectrum information
spectrum_wastewater = data_wastewater[:, 18:1880]
spectrum_wastewater = np.clip(spectrum_wastewater, None, 60000)
spectrum_wastewater[:, zero_index] = 0
spectrum_wastewater = spectrum_wastewater / 60000
Y_pred = TCT_Model.predict([spectrum_wastewater])

# Testing wastewater
plt.plot(range(0, 30), Y_pred[:30], ".", color="orange", alpha=0.7)
plt.plot(range(30, 60), Y_pred[30:60], ".", color="m", alpha=0.7)
plt.axvline(x=60, color="black", linestyle="--", linewidth=1.5)
plt.plot(range(60, 90), Y_pred[60:90], ".", color="orange", alpha=0.7)
plt.plot(range(90, 120), Y_pred[90:120], ".", color="m", alpha=0.7)
plt.axvline(x=120, color="black", linestyle="--", linewidth=1.5)
plt.plot(range(120, 150), Y_pred[120:150], ".", color="orange", alpha=0.7)
plt.plot(range(150, 180), Y_pred[150:180], ".", color="m", alpha=0.7)

plt.ylim(0, 22)
# plt.plot(concentration_wastewater, color='black', linewidth=3.0)
plt.xlabel("number of testing data", fontsize=12)
plt.ylabel("Concentration (ppm)", fontsize=12)
plt.title("Predictions vs Actual Concentrations", fontsize=14)
# Add legend
orange_patch = mpatches.Patch(color="orange", label="Before")
magenta_patch = mpatches.Patch(color="m", label="+5ppm")
blue_patch = mpatches.Patch(color="blue", label="+10ppm")
plt.legend(handles=[orange_patch, magenta_patch, blue_patch], fontsize=10, loc="upper right")
print(tf.keras.losses.MeanSquaredError()(concentration_wastewater, Y_pred))

# Average each group of 30 predictions
for i in range(0, len(Y_pred), 30):
    avg = np.mean(Y_pred[i : i + 30])
    print(f"Y_pred {i}-{i+29} mean: {avg:.3f}")


In [ ]:
import numpy as np

# Average groups of 6 observations to obtain 5 group means
a_mean = Y_pred[90:120].reshape(-1, 6).mean(axis=1)
b_mean = Y_pred[60:90].reshape(-1, 6).mean(axis=1)
c_mean = Y_pred[150:180].reshape(-1, 6).mean(axis=1)
d_mean = Y_pred[120:150].reshape(-1, 6).mean(axis=1)

# Calculate the standard deviation across the 5 group means
a_std = np.std(a_mean)
b_std = np.std(b_mean)
c_std = np.std(c_mean)
d_std = np.std(d_mean)

# Calculate paired differences
diff_ab = a_mean - b_mean
diff_cd = c_mean - d_mean

# Calculate the mean difference
mean_ab = np.mean(diff_ab)
mean_cd = np.mean(diff_cd)

# Propagate uncertainty
std_ab = np.sqrt(a_std**2 + b_std**2)
std_cd = np.sqrt(c_std**2 + d_std**2)

# Display the results
print(f"(a - b) → Mean: {mean_ab:.3f}, Propagated Std: {std_ab:.3f}")
print(f"(c - d) → Mean: {mean_cd:.3f}, Propagated Std: {std_cd:.3f}")
